# 🛡️ Gemini Polyglot Guardian

## Multi-Agent Content Safety System for African Languages

**Powered by LangGraph & Google Gemini API**

---

### 🎯 What This Tool Does

The **Gemini Polyglot Guardian** is a multi-agent content safety analysis system specializing in **African low-resource languages**:

1. **🔍 Language Detection** - Identifies 29+ African languages (Kirundi, Kinyarwanda, Swahili, Yoruba, Hausa, Igbo, Amharic, Zulu, Lingala, Wolof, etc.)
2. **📝 Translation** - Accurate translation to English for analysis
3. **🛡️ Safety Classification** - Categorizes content:
   - ✅ **Safe** - Benign content
   - 📰 **Misinformation** - False claims, health myths
   - 🚨 **Scam** - Mobile money fraud, 419 scams, phishing
   - 🚫 **Hate Speech** - Content promoting hatred
   - 🎭 **Manipulation** - Psychological exploitation
4. **🌍 Cultural Context** - Regional insights for African languages (M-Pesa scams, regional patterns)
5. **⚠️ Risk Assessment** - Comprehensive risk levels with actionable suggestions

---

### 🤖 Multi-Agent Architecture (LangGraph)

```
🔍 Language Detective → 🛡️ Safety Analyzer → 🌍 Cultural Context → ⚠️ Risk Assessor
```

Each specialized agent contributes to the final analysis, providing transparency and accurate results.

---

### 📚 Technologies
- **LangGraph** - Multi-agent orchestration
- **Google Gemini 2.0 Flash** - AI backbone
- **Gradio** - Interactive web interface
- **Pydantic** - Structured outputs

---

**Author:** AI Engineering Team | Hackathon Project  
**License:** MIT

## 📦 Section 1: Setup & Configuration

First, let's import all required libraries and load the API key from the `.env` file.

In [ ]:
# =============================================================================
# IMPORTS & CONFIGURATION
# =============================================================================

import os
import json
from typing import Optional, Annotated, List, Any, Dict, Literal
from dataclasses import dataclass, field
from enum import Enum
from datetime import datetime

# Load environment variables from .env file
from dotenv import load_dotenv
load_dotenv()

# Google GenAI SDK (new package replacing deprecated google-generativeai)
from google import genai

# LangGraph for multi-agent orchestration
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.checkpoint.memory import MemorySaver
from typing_extensions import TypedDict

# Pydantic for structured outputs
from pydantic import BaseModel, Field

# Gradio for web interface
import gradio as gr

# For graph visualization
from IPython.display import Image, display

# Verify API key is loaded
api_key = os.environ.get("GEMINI_API_KEY")
if api_key:
    print("✅ GEMINI_API_KEY loaded successfully!")
    print(f"   Key preview: {api_key[:8]}...{api_key[-4:]}")
else:
    print("❌ ERROR: GEMINI_API_KEY not found in .env file!")
    print("   Please create a .env file with: GEMINI_API_KEY=your-api-key-here")

print("✅ LangGraph multi-agent framework loaded!")

## 🎯 Section 2: Core Components

### Data Classes and Enums

Define the structured types for risk categories, levels, and analysis results.

In [ ]:
# =============================================================================
# DATA CLASSES & ENUMS
# =============================================================================

class RiskCategory(Enum):
    """Content risk categories for classification."""
    SAFE = "Safe"
    MISINFORMATION = "Misinformation"
    SCAM = "Scam"
    HATE_SPEECH = "Hate Speech"
    MANIPULATION = "Manipulation"


class RiskLevel(Enum):
    """Risk severity levels."""
    NONE = "None"
    LOW = "Low"
    MEDIUM = "Medium"
    HIGH = "High"
    CRITICAL = "Critical"


@dataclass
class AnalysisResult:
    """
    Structured result from content analysis.
    
    Attributes:
        language: Detected language of the input text
        translation: English translation (or original if already English)
        category: Classification category (Safe, Scam, etc.)
        risk_level: Severity level (None to Critical)
        explanation: Detailed reasoning for the classification
        suggested_action: Recommended action for the user
        raw_response: Optional raw API response for debugging
    """
    language: str
    translation: str
    category: str
    risk_level: str
    explanation: str
    suggested_action: str
    raw_response: Optional[str] = None
    
    def to_dict(self) -> dict:
        """Convert to dictionary for JSON serialization."""
        return {
            "language": self.language,
            "translation": self.translation,
            "category": self.category,
            "risk_level": self.risk_level,
            "explanation": self.explanation,
            "suggested_action": self.suggested_action
        }
    
    def to_json(self, indent: int = 2) -> str:
        """Convert to formatted JSON string."""
        return json.dumps(self.to_dict(), indent=indent, ensure_ascii=False)
    
    def display(self):
        """Pretty print the analysis result."""
        emoji = get_category_emoji(self.category)
        print(f"\n{'='*60}")
        print(f"{emoji} ANALYSIS RESULT")
        print(f"{'='*60}")
        print(f"\n📍 Language: {self.language}")
        print(f"📊 Category: {self.category}")
        print(f"⚠️  Risk Level: {self.risk_level}")
        print(f"\n📝 Translation:\n   {self.translation}")
        print(f"\n🔍 Explanation:\n   {self.explanation}")
        print(f"\n💡 Suggested Action:\n   {self.suggested_action}")
        print(f"\n{'='*60}\n")


# Helper functions for visualization
def get_risk_color(risk_level: str) -> str:
    """Get a color code for risk level visualization."""
    colors = {
        "None": "#22c55e",      # Green
        "Low": "#84cc16",       # Lime
        "Medium": "#eab308",    # Yellow
        "High": "#f97316",      # Orange
        "Critical": "#ef4444"   # Red
    }
    return colors.get(risk_level, "#6b7280")


def get_category_emoji(category: str) -> str:
    """Get an emoji for category visualization."""
    emojis = {
        "Safe": "✅",
        "Misinformation": "📰",
        "Scam": "🚨",
        "Hate Speech": "🚫",
        "Manipulation": "🎭"
    }
    return emojis.get(category, "❓")


print("✅ Data classes and helper functions defined!")

## 🧠 Section 2B: LangGraph Multi-Agent Architecture

### How LangGraph Works

**LangGraph** is a powerful framework for building stateful, multi-agent applications. Think of it like a road trip:

- **State** = Your car (carries data through the journey)
- **Nodes** = Cities where you stop and do something (each agent)
- **Edges** = Roads connecting cities (flow between agents)
- **Superstep** = One complete round of execution

### Our Multi-Agent System

We use **4 specialized agents** working together:

```
┌─────────────────────────────────────────────────────────────────┐
│                    POLYGLOT GUARDIAN PIPELINE                    │
├─────────────────────────────────────────────────────────────────┤
│                                                                  │
│   START                                                          │
│     │                                                            │
│     ▼                                                            │
│  ┌──────────────────┐                                           │
│  │ 🔍 LANGUAGE      │  Detects language & translates            │
│  │    DETECTIVE     │  (Specializes in African languages)       │
│  └────────┬─────────┘                                           │
│           │                                                      │
│           ▼                                                      │
│  ┌──────────────────┐                                           │
│  │ 🛡️ SAFETY        │  Analyzes content for threats             │
│  │    ANALYZER      │  (Scams, misinformation, hate speech)     │
│  └────────┬─────────┘                                           │
│           │                                                      │
│           ▼                                                      │
│     ┌─────────────┐                                             │
│     │ Is African  │                                              │
│     │ Language?   │                                              │
│     └──────┬──────┘                                             │
│       YES/ \NO                                                   │
│         /   \                                                    │
│        ▼     ▼                                                  │
│  ┌──────────────────┐    ┌──────────────────┐                  │
│  │ 🌍 CULTURAL      │    │ ⚠️ RISK          │                  │
│  │    CONTEXT       │───▶│    ASSESSOR      │                  │
│  │ (African Expert) │    │ (Final Verdict)  │                  │
│  └──────────────────┘    └────────┬─────────┘                  │
│                                   │                              │
│                                   ▼                              │
│                                  END                             │
│                                                                  │
└─────────────────────────────────────────────────────────────────┘
```

### Why Multi-Agent?

1. **Specialization**: Each agent is an expert in its domain
2. **Modularity**: Easy to update or replace individual agents
3. **Transparency**: See the reasoning at each step
4. **Cultural Awareness**: Dedicated agent for African language context

In [ ]:
# =============================================================================
# LANGGRAPH STATE DEFINITION
# =============================================================================

# List of African languages we specialize in
AFRICAN_LANGUAGES = [
    "Kirundi", "Kinyarwanda", "Swahili", "Yoruba", "Hausa", "Igbo",
    "Amharic", "Tigrinya", "Oromo", "Somali", "Zulu", "Xhosa",
    "Shona", "Lingala", "Wolof", "Twi", "Ewe", "Fon", "Bambara",
    "Fulani", "Kanuri", "Tswana", "Sotho", "Ndebele", "Luganda",
    "Kongo", "Chichewa", "Malagasy", "Afrikaans"
]

class GuardianState(TypedDict):
    """
    State that flows through the multi-agent pipeline.
    
    Each agent reads from and writes to this shared state,
    allowing information to accumulate as it flows through the graph.
    """
    # Input
    input_text: str
    language_hint: Optional[str]
    
    # Language Detective outputs
    detected_language: str
    translation: str
    is_african_language: bool
    language_confidence: str
    
    # Safety Analyzer outputs
    safety_category: str
    threat_indicators: List[str]
    safety_reasoning: str
    
    # Cultural Context outputs (for African languages)
    cultural_notes: str
    regional_context: str
    
    # Risk Assessor outputs (final)
    risk_level: str
    final_explanation: str
    suggested_action: str
    
    # Agent message log (for transparency)
    agent_messages: Annotated[List[str], lambda x, y: x + y]


# Pydantic models for structured outputs from each agent
class LanguageDetectiveOutput(BaseModel):
    """Structured output from Language Detective agent."""
    detected_language: str = Field(description="The detected language name")
    translation: str = Field(description="English translation of the text")
    confidence: str = Field(description="Confidence level: High, Medium, or Low")
    is_african: bool = Field(description="Whether this is an African language")


class SafetyAnalyzerOutput(BaseModel):
    """Structured output from Safety Analyzer agent."""
    category: str = Field(description="One of: Safe, Misinformation, Scam, Hate Speech, Manipulation")
    threat_indicators: List[str] = Field(description="List of specific threat indicators found")
    reasoning: str = Field(description="Detailed reasoning for the classification")


class CulturalContextOutput(BaseModel):
    """Structured output from Cultural Context agent."""
    cultural_notes: str = Field(description="Cultural context relevant to understanding the content")
    regional_context: str = Field(description="Regional/country-specific context")
    cultural_risk_factors: str = Field(description="Any culture-specific risk factors to consider")


class RiskAssessorOutput(BaseModel):
    """Structured output from Risk Assessor agent."""
    risk_level: str = Field(description="One of: None, Low, Medium, High, Critical")
    final_explanation: str = Field(description="Comprehensive explanation combining all agent insights")
    suggested_action: str = Field(description="Practical advice for the user")


print("✅ LangGraph State and Pydantic models defined!")
print(f"   Tracking {len(AFRICAN_LANGUAGES)} African languages")

In [ ]:
# =============================================================================
# MULTI-AGENT SYSTEM: THE FOUR GUARDIANS
# =============================================================================

# Initialize the Gemini client (shared across agents)
gemini_client = genai.Client(api_key=os.environ.get("GEMINI_API_KEY"))
# Valid models: "gemini-2.0-flash", "gemini-2.0-flash-lite"
MODEL_NAME = "gemini-2.0-flash"  # Best model - create NEW project if quota exhausted

def call_gemini(prompt: str, temperature: float = 0.3) -> str:
    """Helper function to call Gemini API."""
    response = gemini_client.models.generate_content(
        model=MODEL_NAME,
        contents=prompt,
        config={"temperature": temperature, "max_output_tokens": 2048}
    )
    return response.text


# =============================================================================
# AGENT 1: LANGUAGE DETECTIVE 🔍
# =============================================================================

def language_detective(state: GuardianState) -> Dict[str, Any]:
    """
    🔍 Language Detective Agent
    
    Specializes in detecting languages (especially African low-resource languages)
    and providing accurate translations to English.
    """
    prompt = f"""You are the Language Detective, an expert linguist specializing in African languages.

Your mission: Detect the language and translate the text to English.

INPUT TEXT:
\"\"\"{state['input_text']}\"\"\"

{f"HINT: The user suggests this might be {state['language_hint']}" if state.get('language_hint') else ""}

IMPORTANT: You specialize in these African languages:
{', '.join(AFRICAN_LANGUAGES)}

Respond with ONLY valid JSON (no markdown):
{{
    "detected_language": "<language name>",
    "translation": "<English translation>",
    "confidence": "<High/Medium/Low>",
    "is_african": <true/false>
}}"""

    response = call_gemini(prompt)
    
    try:
        # Parse JSON response
        cleaned = response.strip()
        if cleaned.startswith("```"):
            cleaned = cleaned.split("\n", 1)[1].rsplit("```", 1)[0]
        data = json.loads(cleaned)
        
        return {
            "detected_language": data.get("detected_language", "Unknown"),
            "translation": data.get("translation", state['input_text']),
            "is_african_language": data.get("is_african", False),
            "language_confidence": data.get("confidence", "Medium"),
            "agent_messages": [f"🔍 Language Detective: Detected {data.get('detected_language', 'Unknown')} ({data.get('confidence', 'Medium')} confidence)"]
        }
    except:
        return {
            "detected_language": "Unknown",
            "translation": state['input_text'],
            "is_african_language": False,
            "language_confidence": "Low",
            "agent_messages": ["🔍 Language Detective: Unable to parse response, using original text"]
        }


# =============================================================================
# AGENT 2: SAFETY ANALYZER 🛡️
# =============================================================================

def safety_analyzer(state: GuardianState) -> Dict[str, Any]:
    """
    🛡️ Safety Analyzer Agent
    
    Analyzes content for safety concerns including scams, misinformation,
    hate speech, and manipulation.
    """
    prompt = f"""You are the Safety Analyzer, an expert in identifying harmful content.

ORIGINAL TEXT ({state['detected_language']}):
\"\"\"{state['input_text']}\"\"\"

ENGLISH TRANSLATION:
\"\"\"{state['translation']}\"\"\"

Analyze this content for safety concerns.

CATEGORIES:
1. Safe - Benign content with no concerns
2. Misinformation - False claims, health myths, conspiracy theories
3. Scam - Fraud, phishing, financial exploitation, lottery scams
4. Hate Speech - Content promoting hatred, discrimination
5. Manipulation - Psychological manipulation, coercion

Look for these INDICATORS:
- Urgency language ("Act now!", "Limited time!")
- Requests for personal information or money
- Too-good-to-be-true promises
- Conspiracy language, medical misinformation
- Dehumanizing language, slurs, calls to violence
- Emotional manipulation, guilt-tripping, threats

Respond with ONLY valid JSON:
{{
    "category": "<Safe/Misinformation/Scam/Hate Speech/Manipulation>",
    "threat_indicators": ["<indicator 1>", "<indicator 2>"],
    "reasoning": "<detailed explanation>"
}}"""

    response = call_gemini(prompt)
    
    try:
        cleaned = response.strip()
        if cleaned.startswith("```"):
            cleaned = cleaned.split("\n", 1)[1].rsplit("```", 1)[0]
        data = json.loads(cleaned)
        
        return {
            "safety_category": data.get("category", "Safe"),
            "threat_indicators": data.get("threat_indicators", []),
            "safety_reasoning": data.get("reasoning", "No specific concerns identified"),
            "agent_messages": [f"🛡️ Safety Analyzer: Classified as {data.get('category', 'Safe')} with {len(data.get('threat_indicators', []))} indicators"]
        }
    except:
        return {
            "safety_category": "Safe",
            "threat_indicators": [],
            "safety_reasoning": "Unable to analyze - defaulting to Safe",
            "agent_messages": ["🛡️ Safety Analyzer: Parse error, defaulting to Safe"]
        }


# =============================================================================
# AGENT 3: CULTURAL CONTEXT SPECIALIST 🌍
# =============================================================================

def cultural_context_agent(state: GuardianState) -> Dict[str, Any]:
    """
    🌍 Cultural Context Agent
    
    Provides cultural context for African languages and regions.
    Only activated for African language content.
    """
    prompt = f"""You are the Cultural Context Specialist, an expert in African cultures and languages.

LANGUAGE: {state['detected_language']}
ORIGINAL TEXT: \"\"\"{state['input_text']}\"\"\"
TRANSLATION: \"\"\"{state['translation']}\"\"\"
INITIAL SAFETY ASSESSMENT: {state['safety_category']}

Provide cultural context that might affect understanding:

1. What region/country does this language primarily come from?
2. Are there cultural norms that affect interpretation?
3. Are there culture-specific scam patterns or concerns in this region?
4. Any phrases or idioms that might be misinterpreted?

Examples of regional scam patterns:
- Mobile money scams (M-Pesa, Orange Money, MTN Mobile Money)
- SIM swap scams
- Fake lottery/competition wins
- "Urgent" money transfer requests

Respond with ONLY valid JSON:
{{
    "cultural_notes": "<cultural context that affects understanding>",
    "regional_context": "<country/region and relevant information>",
    "cultural_risk_factors": "<any culture-specific risk factors>"
}}"""

    response = call_gemini(prompt)
    
    try:
        cleaned = response.strip()
        if cleaned.startswith("```"):
            cleaned = cleaned.split("\n", 1)[1].rsplit("```", 1)[0]
        data = json.loads(cleaned)
        
        return {
            "cultural_notes": data.get("cultural_notes", "No specific cultural notes"),
            "regional_context": data.get("regional_context", "African region"),
            "agent_messages": [f"🌍 Cultural Context: Added regional insights for {state['detected_language']}"]
        }
    except:
        return {
            "cultural_notes": "Unable to provide cultural context",
            "regional_context": "Unknown region",
            "agent_messages": ["🌍 Cultural Context: Parse error"]
        }


# =============================================================================
# AGENT 4: RISK ASSESSOR 🎯
# =============================================================================

def risk_assessor(state: GuardianState) -> Dict[str, Any]:
    """
    ⚠️ Risk Assessor Agent
    
    Makes final risk assessment combining all previous agent insights.
    Provides comprehensive explanation and actionable suggestions.
    """
    cultural_info = ""
    if state.get('is_african_language') and state.get('cultural_notes'):
        cultural_info = f"""
CULTURAL CONTEXT (African Language):
- Notes: {state.get('cultural_notes', 'N/A')}
- Region: {state.get('regional_context', 'N/A')}
"""

    prompt = f"""You are the Risk Assessor, making the FINAL safety determination.

LANGUAGE ANALYSIS:
- Detected: {state['detected_language']}
- Confidence: {state.get('language_confidence', 'Medium')}
- African Language: {state.get('is_african_language', False)}

TRANSLATION:
\"\"\"{state['translation']}\"\"\"

SAFETY ANALYSIS:
- Category: {state['safety_category']}
- Threat Indicators: {', '.join(state.get('threat_indicators', [])) or 'None'}
- Reasoning: {state.get('safety_reasoning', 'N/A')}
{cultural_info}

Based on ALL the above, provide your FINAL assessment.

RISK LEVELS:
- None: Completely safe, no concerns
- Low: Minor issues, generally safe
- Medium: Moderate concern, caution advised
- High: Significant risk, intervention recommended
- Critical: Severe threat, immediate action needed

Respond with ONLY valid JSON:
{{
    "risk_level": "<None/Low/Medium/High/Critical>",
    "final_explanation": "<comprehensive explanation combining all insights>",
    "suggested_action": "<specific, practical advice for the user>"
}}"""

    response = call_gemini(prompt)
    
    try:
        cleaned = response.strip()
        if cleaned.startswith("```"):
            cleaned = cleaned.split("\n", 1)[1].rsplit("```", 1)[0]
        data = json.loads(cleaned)
        
        return {
            "risk_level": data.get("risk_level", "None"),
            "final_explanation": data.get("final_explanation", "No explanation available"),
            "suggested_action": data.get("suggested_action", "No specific action required"),
            "agent_messages": [f"⚠️ Risk Assessor: Final verdict - {data.get('risk_level', 'None')} risk"]
        }
    except:
        return {
            "risk_level": "None",
            "final_explanation": "Unable to complete assessment",
            "suggested_action": "Please try again",
            "agent_messages": ["⚠️ Risk Assessor: Parse error"]
        }


print("✅ All 4 Guardian Agents defined!")
print("   🔍 Language Detective - Detects & translates (African language expert)")
print("   🛡️ Safety Analyzer - Identifies threats & harmful content")
print("   🌍 Cultural Context - African cultural insights")
print("   ⚠️ Risk Assessor - Final verdict & recommendations")

In [ ]:
# =============================================================================
# LANGGRAPH WORKFLOW BUILDER
# =============================================================================

def route_after_safety(state: GuardianState) -> Literal["cultural_context", "risk_assessor"]:
    """
    Routing function: Decides whether to add cultural context.
    
    If the detected language is African, we route to the Cultural Context agent
    for additional regional insights before the final assessment.
    """
    if state.get("is_african_language", False):
        print("   → Routing to Cultural Context (African language detected)")
        return "cultural_context"
    else:
        print("   → Skipping Cultural Context (non-African language)")
        return "risk_assessor"


def build_guardian_graph():
    """
    Build the LangGraph multi-agent workflow.
    
    Flow:
    START → Language Detective → Safety Analyzer → [Cultural Context?] → Risk Assessor → END
    
    The Cultural Context agent is only called for African languages.
    """
    # Create the graph with our state
    graph_builder = StateGraph(GuardianState)
    
    # Add nodes (agents)
    graph_builder.add_node("language_detective", language_detective)
    graph_builder.add_node("safety_analyzer", safety_analyzer)
    graph_builder.add_node("cultural_context", cultural_context_agent)
    graph_builder.add_node("risk_assessor", risk_assessor)
    
    # Add edges (flow)
    graph_builder.add_edge(START, "language_detective")
    graph_builder.add_edge("language_detective", "safety_analyzer")
    
    # Conditional routing after safety analyzer
    graph_builder.add_conditional_edges(
        "safety_analyzer",
        route_after_safety,
        {
            "cultural_context": "cultural_context",
            "risk_assessor": "risk_assessor"
        }
    )
    
    # Cultural context leads to risk assessor
    graph_builder.add_edge("cultural_context", "risk_assessor")
    
    # Risk assessor is the final step
    graph_builder.add_edge("risk_assessor", END)
    
    # Compile with memory for conversation tracking
    memory = MemorySaver()
    graph = graph_builder.compile(checkpointer=memory)
    
    return graph


# Build the graph
guardian_graph = build_guardian_graph()

print("✅ LangGraph workflow built!")
print("   Flow: START → Language Detective → Safety Analyzer → [Cultural Context?] → Risk Assessor → END")

In [ ]:
# =============================================================================
# VISUALIZE THE GRAPH
# =============================================================================

print("📊 Multi-Agent Graph Visualization:")
print("=" * 60)

try:
    # Display the graph using Mermaid
    graph_image = guardian_graph.get_graph().draw_mermaid_png()
    display(Image(graph_image))
except Exception as e:
    print(f"   (Graph visualization requires graphviz: {e})")
    print("""
    Manual Graph Structure:
    
    ┌─────────────────┐
    │      START      │
    └────────┬────────┘
             │
             ▼
    ┌─────────────────┐
    │    Language     │
    │    Detective    │
    └────────┬────────┘
             │
             ▼
    ┌─────────────────┐
    │     Safety      │
    │    Analyzer     │
    └────────┬────────┘
             │
       ┌─────┴─────┐
       │ African?  │
       └─────┬─────┘
        YES /  \\ NO
           /    \\
          ▼      ▼
    ┌─────────┐  │
    │Cultural │  │
    │Context  │──┘
    └────┬────┘
         │
         ▼
    ┌─────────────────┐
    │      Risk       │
    │    Assessor     │
    └────────┬────────┘
             │
             ▼
    ┌─────────────────┐
    │       END       │
    └─────────────────┘
    """)

In [ ]:
# =============================================================================
# MULTI-AGENT ANALYSIS FUNCTION
# =============================================================================

import uuid

def analyze_with_agents(
    text: str, 
    language_hint: Optional[str] = None,
    verbose: bool = True
) -> Dict[str, Any]:
    """
    Run the multi-agent pipeline to analyze text.
    
    Args:
        text: Input text to analyze (any language)
        language_hint: Optional hint about the language
        verbose: Whether to print agent messages
    
    Returns:
        Dictionary with complete analysis results
    """
    if verbose:
        print(f"\n{'='*60}")
        print("🛡️ POLYGLOT GUARDIAN - Multi-Agent Analysis")
        print(f"{'='*60}")
        print(f"📝 Input: {text[:100]}{'...' if len(text) > 100 else ''}")
        print(f"{'='*60}\n")
    
    # Create initial state
    initial_state = {
        "input_text": text,
        "language_hint": language_hint,
        "detected_language": "",
        "translation": "",
        "is_african_language": False,
        "language_confidence": "",
        "safety_category": "",
        "threat_indicators": [],
        "safety_reasoning": "",
        "cultural_notes": "",
        "regional_context": "",
        "risk_level": "",
        "final_explanation": "",
        "suggested_action": "",
        "agent_messages": []
    }
    
    # Create unique thread ID for this analysis
    config = {"configurable": {"thread_id": str(uuid.uuid4())}}
    
    # Run the graph
    if verbose:
        print("🚀 Starting multi-agent pipeline...\n")
    
    result = guardian_graph.invoke(initial_state, config=config)
    
    # Print agent messages if verbose
    if verbose:
        print("\n📋 Agent Activity Log:")
        print("-" * 40)
        for msg in result.get("agent_messages", []):
            print(f"   {msg}")
        print("-" * 40)
        
        # Print summary
        print(f"\n{'='*60}")
        print("📊 FINAL ANALYSIS RESULTS")
        print(f"{'='*60}")
        emoji = get_category_emoji(result.get("safety_category", "Safe"))
        print(f"\n🌐 Language: {result.get('detected_language', 'Unknown')}")
        print(f"   African: {'Yes 🌍' if result.get('is_african_language') else 'No'}")
        print(f"   Confidence: {result.get('language_confidence', 'Unknown')}")
        print(f"\n📝 Translation: {result.get('translation', 'N/A')[:200]}...")
        print(f"\n{emoji} Category: {result.get('safety_category', 'Unknown')}")
        print(f"⚠️  Risk Level: {result.get('risk_level', 'Unknown')}")
        
        if result.get('threat_indicators'):
            print(f"\n🚩 Threat Indicators:")
            for indicator in result.get('threat_indicators', []):
                print(f"   • {indicator}")
        
        if result.get('cultural_notes') and result.get('is_african_language'):
            print(f"\n🌍 Cultural Context:")
            print(f"   Region: {result.get('regional_context', 'N/A')}")
            print(f"   Notes: {result.get('cultural_notes', 'N/A')[:200]}...")
        
        print(f"\n📖 Explanation:")
        print(f"   {result.get('final_explanation', 'N/A')}")
        print(f"\n💡 Suggested Action:")
        print(f"   {result.get('suggested_action', 'N/A')}")
        print(f"\n{'='*60}\n")
    
    return result


# Create a result dataclass from multi-agent output
def result_from_agents(agent_result: Dict[str, Any]) -> AnalysisResult:
    """Convert multi-agent result to AnalysisResult dataclass."""
    return AnalysisResult(
        language=agent_result.get("detected_language", "Unknown"),
        translation=agent_result.get("translation", ""),
        category=agent_result.get("safety_category", "Safe"),
        risk_level=agent_result.get("risk_level", "None"),
        explanation=agent_result.get("final_explanation", ""),
        suggested_action=agent_result.get("suggested_action", ""),
        raw_response=json.dumps({
            "agent_messages": agent_result.get("agent_messages", []),
            "threat_indicators": agent_result.get("threat_indicators", []),
            "cultural_notes": agent_result.get("cultural_notes", ""),
            "regional_context": agent_result.get("regional_context", "")
        }, indent=2)
    )


print("✅ Multi-agent analysis function ready!")
print("   Use: analyze_with_agents(text, language_hint=None, verbose=True)")

## 🌍 Section 3: African Language Test Cases

### Comprehensive Test Suite for Low-Resource African Languages

We test the following languages from different regions:

| Language | Country/Region | Language Family |
|----------|----------------|-----------------|
| 🇧🇮 Kirundi | Burundi | Bantu |
| 🇷🇼 Kinyarwanda | Rwanda | Bantu |
| 🇰🇪🇹🇿 Swahili | East Africa | Bantu |
| 🇳🇬 Yoruba | Nigeria | Niger-Congo |
| 🇳🇬 Hausa | Nigeria/Niger | Afroasiatic |
| 🇳🇬 Igbo | Nigeria | Niger-Congo |
| 🇪🇹 Amharic | Ethiopia | Semitic |
| 🇿🇦 Zulu | South Africa | Bantu |
| 🇨🇩 Lingala | Congo | Bantu |
| 🇸🇳 Wolof | Senegal | Atlantic |

Each test case includes:
- **Safe content** (greetings, everyday phrases)
- **Scam patterns** (mobile money scams, lottery fraud)
- **Misinformation** (health myths, conspiracy theories)
- **Manipulation** (emotional exploitation)

In [ ]:
# =============================================================================
# COMPREHENSIVE AFRICAN LANGUAGE TEST CASES
# =============================================================================

AFRICAN_TEST_CASES = {
    # =========================================================================
    # KIRUNDI (Burundi) 🇧🇮
    # =========================================================================
    "kirundi_safe_greeting": {
        "text": "Mwiriwe neza! Ndagukunda cane. Umunsi mwiza! Amahoro.",
        "language": "Kirundi",
        "description": "Friendly greeting: 'Good evening! I love you very much. Have a nice day! Peace.'",
        "expected_category": "Safe",
        "region": "Burundi"
    },
    "kirundi_scam_money": {
        "text": "Watsinze amafaranga menshi! Tuma nomero yawe ya telefone n'amafaranga 5000 kuri MTN Mobile Money ubu nyene kugira uronke igihembo cawe!",
        "language": "Kirundi",
        "description": "Mobile money scam: 'You won a lot of money! Send your phone number and 5000 francs via MTN Mobile Money now to receive your prize!'",
        "expected_category": "Scam",
        "region": "Burundi"
    },
    
    # =========================================================================
    # KINYARWANDA (Rwanda) 🇷🇼
    # =========================================================================
    "kinyarwanda_safe_greeting": {
        "text": "Muraho neza! Amakuru? Ndashimira Imana kubona mwese. Umunsi mwiza!",
        "language": "Kinyarwanda",
        "description": "Friendly greeting: 'Hello! How are you? I thank God to see you all. Have a nice day!'",
        "expected_category": "Safe",
        "region": "Rwanda"
    },
    "kinyarwanda_scam_lottery": {
        "text": "WATSINDIYE MILIYONI 10! Ohereza ubutumwa bukubiyemo nomero yawe ya konti na PIN yawe kuri 2255 ubu! Iyi ni tombola y'ukuri!",
        "language": "Kinyarwanda",
        "description": "Lottery scam: 'YOU WON 10 MILLION! Send your account number and PIN to 2255 now! This is a real lottery!'",
        "expected_category": "Scam",
        "region": "Rwanda"
    },
    "kinyarwanda_misinfo_health": {
        "text": "Urukingo rwa COVID rugira chip ya 5G yo kukugenzura. Leta irabyihisha. Sabagura ubu mbere yuko bihanagurwa!",
        "language": "Kinyarwanda",
        "description": "Misinformation: 'The COVID vaccine has a 5G chip to control you. The government is hiding it. Share before they delete it!'",
        "expected_category": "Misinformation",
        "region": "Rwanda"
    },
    
    # =========================================================================
    # SWAHILI (Kenya, Tanzania, East Africa) 🇰🇪🇹🇿
    # =========================================================================
    "swahili_safe_greeting": {
        "text": "Habari yako! Karibu sana. Nakutakia siku njema na baraka nyingi.",
        "language": "Swahili",
        "description": "Friendly greeting: 'How are you! Welcome very much. I wish you a good day and many blessings.'",
        "expected_category": "Safe",
        "region": "East Africa"
    },
    "swahili_scam_mpesa": {
        "text": "Umeshinda shilingi milioni 10! Tuma nambari yako ya simu na malipo ya usajili wa shilingi 5000 kupitia M-Pesa sasa hivi! Fursa hii itaisha leo!",
        "language": "Swahili",
        "description": "M-Pesa scam: 'You won 10 million shillings! Send your phone number and 5000 shillings registration fee via M-Pesa now! This opportunity ends today!'",
        "expected_category": "Scam",
        "region": "Kenya/Tanzania"
    },
    "swahili_misinfo_health": {
        "text": "Chanjo ya COVID ina sumu inayoua watu baada ya miaka miwili. Madaktari wanajua lakini wanaogopa kusema. Shiriki kabla hawajaifuta!",
        "language": "Swahili",
        "description": "Health misinformation: 'The COVID vaccine has poison that kills people after two years. Doctors know but are afraid to say. Share before they delete it!'",
        "expected_category": "Misinformation",
        "region": "East Africa"
    },
    
    # =========================================================================
    # YORUBA (Nigeria) 🇳🇬
    # =========================================================================
    "yoruba_safe_greeting": {
        "text": "E kaaro o! Bawo ni? Mo dupé lọwọ Ọlọrun pe a wa laaye. Ọjọ rere!",
        "language": "Yoruba",
        "description": "Friendly greeting: 'Good morning! How are you? I thank God that we are alive. Good day!'",
        "expected_category": "Safe",
        "region": "Nigeria (Southwest)"
    },
    "yoruba_scam_419": {
        "text": "O ti gba $5,000,000 lati inu ọrọ ogún! Fi nọmba àkọọlẹ̀ banki rẹ ati koodu PIN ranṣẹ si mi lati gba owo rẹ. Eyi jẹ gidi!",
        "language": "Yoruba",
        "description": "419 scam: 'You have received $5,000,000 from an inheritance! Send me your bank account number and PIN code to receive your money. This is real!'",
        "expected_category": "Scam",
        "region": "Nigeria"
    },
    
    # =========================================================================
    # HAUSA (Nigeria, Niger) 🇳🇬
    # =========================================================================
    "hausa_safe_greeting": {
        "text": "Sannu da zuwa! Yaya aiki? Allah ya ba mu lafiya. Rana mai kyau!",
        "language": "Hausa",
        "description": "Friendly greeting: 'Welcome! How is work? May God give us health. Good day!'",
        "expected_category": "Safe",
        "region": "Nigeria (North)/Niger"
    },
    "hausa_scam_lottery": {
        "text": "KA CI NASARA! Ka samu naira miliyan 50! Aika lambar wayarka da kudin rajista N10,000 zuwa wannan lamba YANZU! Wannan gaske ne!",
        "language": "Hausa",
        "description": "Lottery scam: 'YOU WON! You got 50 million naira! Send your phone number and N10,000 registration fee to this number NOW! This is real!'",
        "expected_category": "Scam",
        "region": "Nigeria/Niger"
    },
    
    # =========================================================================
    # IGBO (Nigeria) 🇳🇬
    # =========================================================================
    "igbo_safe_greeting": {
        "text": "Nnọọ! Kedu ka ị mere? Chukwu gozie gị. Ụbọchị ọma!",
        "language": "Igbo",
        "description": "Friendly greeting: 'Welcome! How are you? God bless you. Good day!'",
        "expected_category": "Safe",
        "region": "Nigeria (Southeast)"
    },
    "igbo_manipulation": {
        "text": "Ọ bụrụ na ị hụrụ m n'anya, ị ga-ezitere m ego. Ndị ezinụlọ m na-anwụ anwụ ma ọ bụ naanị gị nwere ike inyere anyị aka. Ziga N50,000 ugbu a!",
        "language": "Igbo",
        "description": "Emotional manipulation: 'If you love me, you will send me money. My family is dying and only you can help us. Send N50,000 now!'",
        "expected_category": "Manipulation",
        "region": "Nigeria"
    },
    
    # =========================================================================
    # AMHARIC (Ethiopia) 🇪🇹
    # =========================================================================
    "amharic_safe_greeting": {
        "text": "ሰላም! እንደምን ነህ? እግዚአብሔር ይባርክህ። መልካም ቀን!",
        "language": "Amharic",
        "description": "Friendly greeting: 'Hello! How are you? God bless you. Good day!'",
        "expected_category": "Safe",
        "region": "Ethiopia"
    },
    "amharic_scam_lottery": {
        "text": "አሸንፈሃል! 1,000,000 ብር አግኝተሃል! ስልክ ቁጥርህን እና 5,000 ብር ምዝገባ ክፍያ በቴሌብር ላክ አሁኑኑ!",
        "language": "Amharic",
        "description": "Lottery scam: 'You won! You got 1,000,000 birr! Send your phone number and 5,000 birr registration fee via Telebirr now!'",
        "expected_category": "Scam",
        "region": "Ethiopia"
    },
    
    # =========================================================================
    # ZULU (South Africa) 🇿🇦
    # =========================================================================
    "zulu_safe_greeting": {
        "text": "Sawubona! Unjani? UNkulunkulu akubusise. Usuku oluhle!",
        "language": "Zulu",
        "description": "Friendly greeting: 'Hello! How are you? God bless you. Good day!'",
        "expected_category": "Safe",
        "region": "South Africa"
    },
    "zulu_scam_lottery": {
        "text": "UWINE! Uthole R5,000,000! Thumela inombolo yakho yocingo nenani lokubhalisa R500 nge-EFT manje! Leli thuba liphela namuhla!",
        "language": "Zulu",
        "description": "Lottery scam: 'YOU WON! You got R5,000,000! Send your phone number and R500 registration fee via EFT now! This opportunity ends today!'",
        "expected_category": "Scam",
        "region": "South Africa"
    },
    
    # =========================================================================
    # LINGALA (Congo) 🇨🇩
    # =========================================================================
    "lingala_safe_greeting": {
        "text": "Mbote! Ozali malamu? Nzambe apambola yo. Mokolo malamu!",
        "language": "Lingala",
        "description": "Friendly greeting: 'Hello! Are you well? God bless you. Good day!'",
        "expected_category": "Safe",
        "region": "DR Congo"
    },
    "lingala_scam_money": {
        "text": "Olongi! Ozwi dollar 100,000! Tinda nimero ya telefone na yo na mbongo 50$ na Orange Money sik'oyo! Likambo oyo ezali ya solo!",
        "language": "Lingala",
        "description": "Money scam: 'You won! You got $100,000! Send your phone number and $50 via Orange Money now! This is real!'",
        "expected_category": "Scam",
        "region": "DR Congo"
    },
    
    # =========================================================================
    # WOLOF (Senegal) 🇸🇳
    # =========================================================================
    "wolof_safe_greeting": {
        "text": "Na nga def! Jàmm rekk? Yàlla na la barkeel. Bés bu baax!",
        "language": "Wolof",
        "description": "Friendly greeting: 'How are you! Only peace? May God bless you. Good day!'",
        "expected_category": "Safe",
        "region": "Senegal"
    },
    "wolof_scam_lottery": {
        "text": "DANGAY WÀÑÑI! Jot nga 10,000,000 CFA! Yónnee sa nimero telefon ak 5,000 CFA ci Wave léegi! Lii dëgg la!",
        "language": "Wolof",
        "description": "Lottery scam: 'YOU WON! You got 10,000,000 CFA! Send your phone number and 5,000 CFA via Wave now! This is true!'",
        "expected_category": "Scam",
        "region": "Senegal"
    },
}

# Display summary
print("🌍 African Language Test Cases Loaded!")
print("=" * 60)
print(f"   Total test cases: {len(AFRICAN_TEST_CASES)}")
print("\n   By Language:")
languages = {}
for key, case in AFRICAN_TEST_CASES.items():
    lang = case["language"]
    languages[lang] = languages.get(lang, 0) + 1
for lang, count in sorted(languages.items()):
    print(f"   • {lang}: {count} cases")

print("\n   By Category:")
categories = {}
for key, case in AFRICAN_TEST_CASES.items():
    cat = case["expected_category"]
    categories[cat] = categories.get(cat, 0) + 1
for cat, count in sorted(categories.items()):
    print(f"   • {cat}: {count} cases")

## 🌐 Section 4: Multi-Agent Gradio Interface

### Enhanced UI with Agent Activity Visualization

The updated Gradio interface now:
- Uses the **LangGraph multi-agent pipeline**
- Shows **agent activity log** for transparency
- Displays **cultural context** for African languages
- Provides **threat indicators** visualization
- Includes all **African language examples** as quick buttons

In [ ]:
# =============================================================================
# MULTI-AGENT GRADIO INTERFACE
# =============================================================================

# Enhanced CSS for multi-agent UI
MULTI_AGENT_CSS = """
@import url('https://fonts.googleapis.com/css2?family=Outfit:wght@300;400;500;600;700&family=JetBrains+Mono:wght@400;500&display=swap');

:root {
    --guardian-bg: #0a0e17;
    --guardian-surface: #111827;
    --guardian-border: #1e3a5f;
    --guardian-primary: #00d4ff;
    --guardian-text: #e2e8f0;
    --gradient-cyber: linear-gradient(135deg, #00d4ff 0%, #7c3aed 50%, #f43f5e 100%);
    --gradient-africa: linear-gradient(135deg, #22c55e 0%, #eab308 50%, #ef4444 100%);
}

.gradio-container {
    font-family: 'Outfit', sans-serif !important;
    background: var(--guardian-bg) !important;
}

.guardian-title {
    font-family: 'Outfit', sans-serif;
    font-weight: 700;
    font-size: 2.5rem;
    background: var(--gradient-cyber);
    -webkit-background-clip: text;
    -webkit-text-fill-color: transparent;
    text-align: center;
}

.agent-log {
    background: #1a1a2e;
    border-radius: 8px;
    padding: 12px;
    font-family: 'JetBrains Mono', monospace;
    font-size: 0.9rem;
}

.african-badge {
    background: var(--gradient-africa);
    padding: 4px 12px;
    border-radius: 20px;
    font-weight: 600;
    color: white;
}
"""


def analyze_multiagent_for_gradio(text: str, language_hint: str = "Auto-detect") -> tuple:
    """
    Multi-agent analysis wrapper for Gradio.
    
    Returns:
        Tuple of (summary_markdown, agent_log, json_output)
    """
    if not text or not text.strip():
        return (
            "⚠️ Please enter some text to analyze.",
            "No agents activated yet.",
            json.dumps({"error": "Empty input"}, indent=2)
        )
    
    try:
        # Process language hint
        hint = None if language_hint == "Auto-detect" else language_hint
        
        # Run multi-agent analysis
        result = analyze_with_agents(text, language_hint=hint, verbose=False)
        
        # Format agent log
        agent_log = "### 🤖 Agent Activity Log\n\n"
        for msg in result.get("agent_messages", []):
            agent_log += f"- {msg}\n"
        
        # Format summary
        emoji = get_category_emoji(result.get("safety_category", "Safe"))
        african_badge = "🌍 **African Language Detected**" if result.get("is_african_language") else ""
        
        summary = f"""
## {emoji} Analysis Complete

{african_badge}

| Field | Value |
|-------|-------|
| **Language** | {result.get('detected_language', 'Unknown')} |
| **Confidence** | {result.get('language_confidence', 'Unknown')} |
| **Category** | {result.get('safety_category', 'Unknown')} |
| **Risk Level** | {result.get('risk_level', 'Unknown')} |

---

### 📝 Translation
{result.get('translation', 'N/A')}

---

### 🔍 Analysis
{result.get('final_explanation', 'N/A')}
"""
        
        # Add threat indicators if present
        if result.get('threat_indicators'):
            summary += "\n### 🚩 Threat Indicators\n"
            for indicator in result.get('threat_indicators', []):
                summary += f"- {indicator}\n"
        
        # Add cultural context if African language
        if result.get('is_african_language') and result.get('cultural_notes'):
            summary += f"""
---

### 🌍 Cultural Context
**Region:** {result.get('regional_context', 'N/A')}

{result.get('cultural_notes', 'N/A')}
"""
        
        summary += f"""
---

### 💡 Suggested Action
{result.get('suggested_action', 'N/A')}
"""
        
        # Format JSON output
        json_output = json.dumps({
            "language": result.get("detected_language"),
            "is_african_language": result.get("is_african_language"),
            "translation": result.get("translation"),
            "category": result.get("safety_category"),
            "risk_level": result.get("risk_level"),
            "threat_indicators": result.get("threat_indicators", []),
            "cultural_notes": result.get("cultural_notes", ""),
            "regional_context": result.get("regional_context", ""),
            "explanation": result.get("final_explanation"),
            "suggested_action": result.get("suggested_action"),
            "agent_messages": result.get("agent_messages", [])
        }, indent=2, ensure_ascii=False)
        
        return summary, agent_log, json_output
        
    except Exception as e:
        return (
            f"❌ Error: {str(e)}",
            "Error occurred during analysis",
            json.dumps({"error": str(e)}, indent=2)
        )


def create_multiagent_gradio_app():
    """Create the multi-agent Gradio application."""
    
    with gr.Blocks(
        css=MULTI_AGENT_CSS,
        title="Gemini Polyglot Guardian - Multi-Agent",
        theme=gr.themes.Base(
            primary_hue="cyan",
            secondary_hue="purple",
            neutral_hue="slate",
        )
    ) as app:
        
        # Header
        gr.HTML("""
            <div style="text-align: center; padding: 20px;">
                <div style="font-size: 3rem;">🛡️</div>
                <h1 class="guardian-title">Gemini Polyglot Guardian</h1>
                <p style="color: #94a3b8; max-width: 700px; margin: 10px auto;">
                    <strong>Multi-Agent Content Safety Analysis</strong> powered by LangGraph & Google Gemini.<br/>
                    Specialized in <span class="african-badge">African Languages</span> including Kirundi, Kinyarwanda, Swahili, Yoruba, and more.
                </p>
                <p style="color: #64748b; font-size: 0.9rem;">
                    🔍 Language Detective → 🛡️ Safety Analyzer → 🌍 Cultural Context → ⚠️ Risk Assessor
                </p>
            </div>
        """)
        
        with gr.Row():
            # Left Column - Input
            with gr.Column(scale=1):
                gr.Markdown("### 📥 Input")
                
                input_text = gr.Textbox(
                    label="Text to Analyze",
                    placeholder="Paste or type text in any language...\n\nTry African languages like Kirundi, Kinyarwanda, Swahili, Yoruba, Hausa, Amharic, Zulu, Lingala, or Wolof!",
                    lines=6,
                )
                
                language_dropdown = gr.Dropdown(
                    label="Language Hint (Optional)",
                    choices=[
                        "Auto-detect",
                        "--- African Languages ---",
                        "Kirundi", "Kinyarwanda", "Swahili", "Yoruba", "Hausa", "Igbo",
                        "Amharic", "Zulu", "Lingala", "Wolof", "Shona", "Twi",
                        "--- Other Languages ---",
                        "English", "French", "Spanish", "Arabic", "Portuguese",
                        "Chinese", "Hindi", "Russian", "Japanese", "Korean", "German"
                    ],
                    value="Auto-detect",
                )
                
                analyze_btn = gr.Button("🔍 Analyze with Multi-Agent Pipeline", variant="primary", size="lg")
                
                # African Language Examples
                gr.Markdown("### 🌍 African Language Examples")
                with gr.Row():
                    btn_kirundi = gr.Button("🇧🇮 Kirundi", size="sm")
                    btn_rwanda = gr.Button("🇷🇼 Kinyarwanda", size="sm")
                    btn_swahili = gr.Button("🇰🇪 Swahili", size="sm")
                with gr.Row():
                    btn_yoruba = gr.Button("🇳🇬 Yoruba", size="sm")
                    btn_hausa = gr.Button("🇳🇬 Hausa", size="sm")
                    btn_igbo = gr.Button("🇳🇬 Igbo", size="sm")
                with gr.Row():
                    btn_amharic = gr.Button("🇪🇹 Amharic", size="sm")
                    btn_zulu = gr.Button("🇿🇦 Zulu", size="sm")
                    btn_lingala = gr.Button("🇨🇩 Lingala", size="sm")
                
                # Scam Examples
                gr.Markdown("### 🚨 Scam Examples")
                with gr.Row():
                    btn_scam_kirundi = gr.Button("🇧🇮 Kirundi Scam", size="sm")
                    btn_scam_swahili = gr.Button("🇰🇪 M-Pesa Scam", size="sm")
                    btn_scam_yoruba = gr.Button("🇳🇬 419 Scam", size="sm")
            
            # Right Column - Results
            with gr.Column(scale=1):
                gr.Markdown("### 📊 Analysis Results")
                
                summary_output = gr.Markdown(value="*Results will appear here after analysis...*")
                
                with gr.Accordion("🤖 Agent Activity Log", open=True):
                    agent_log_output = gr.Markdown(value="*No agents activated yet*")
                
                with gr.Accordion("📄 Raw JSON Response", open=False):
                    json_output = gr.Code(label="JSON", language="json", lines=15)
        
        # Footer
        gr.HTML("""
            <div style="text-align: center; padding: 15px; color: #64748b; border-top: 1px solid #1e3a5f; margin-top: 20px;">
                <p>🛡️ Gemini Polyglot Guardian | Multi-Agent System with LangGraph | Powered by Google Gemini</p>
                <p style="font-size: 0.8rem;">Specializing in African Low-Resource Languages: Kirundi, Kinyarwanda, Swahili, Yoruba, Hausa, Igbo, Amharic, Zulu, Lingala, Wolof</p>
            </div>
        """)
        
        # Event handlers
        analyze_btn.click(
            fn=analyze_multiagent_for_gradio,
            inputs=[input_text, language_dropdown],
            outputs=[summary_output, agent_log_output, json_output]
        )
        
        input_text.submit(
            fn=analyze_multiagent_for_gradio,
            inputs=[input_text, language_dropdown],
            outputs=[summary_output, agent_log_output, json_output]
        )
        
        # Safe greeting example handlers
        btn_kirundi.click(lambda: AFRICAN_TEST_CASES["kirundi_safe_greeting"]["text"], outputs=input_text)
        btn_rwanda.click(lambda: AFRICAN_TEST_CASES["kinyarwanda_safe_greeting"]["text"], outputs=input_text)
        btn_swahili.click(lambda: AFRICAN_TEST_CASES["swahili_safe_greeting"]["text"], outputs=input_text)
        btn_yoruba.click(lambda: AFRICAN_TEST_CASES["yoruba_safe_greeting"]["text"], outputs=input_text)
        btn_hausa.click(lambda: AFRICAN_TEST_CASES["hausa_safe_greeting"]["text"], outputs=input_text)
        btn_igbo.click(lambda: AFRICAN_TEST_CASES["igbo_safe_greeting"]["text"], outputs=input_text)
        btn_amharic.click(lambda: AFRICAN_TEST_CASES["amharic_safe_greeting"]["text"], outputs=input_text)
        btn_zulu.click(lambda: AFRICAN_TEST_CASES["zulu_safe_greeting"]["text"], outputs=input_text)
        btn_lingala.click(lambda: AFRICAN_TEST_CASES["lingala_safe_greeting"]["text"], outputs=input_text)
        
        # Scam example handlers
        btn_scam_kirundi.click(lambda: AFRICAN_TEST_CASES["kirundi_scam_money"]["text"], outputs=input_text)
        btn_scam_swahili.click(lambda: AFRICAN_TEST_CASES["swahili_scam_mpesa"]["text"], outputs=input_text)
        btn_scam_yoruba.click(lambda: AFRICAN_TEST_CASES["yoruba_scam_419"]["text"], outputs=input_text)
    
    return app


print("✅ Multi-agent Gradio interface defined!")
print("   Features:")
print("   • LangGraph multi-agent pipeline")
print("   • Agent activity log visualization")
print("   • Cultural context for African languages")
print("   • 10+ African language quick examples")

In [ ]:
# =============================================================================
# LAUNCH MULTI-AGENT GRADIO APP
# =============================================================================

print("🚀 Launching Gemini Polyglot Guardian - Multi-Agent Edition...")
print("=" * 70)
print("\n🤖 Multi-Agent Pipeline:")
print("   1. 🔍 Language Detective - Detects language & translates")
print("   2. 🛡️ Safety Analyzer - Identifies threats")
print("   3. 🌍 Cultural Context - African language insights")
print("   4. ⚠️ Risk Assessor - Final verdict")
print("\n📌 The app will open in a new browser tab.")
print("📌 A public shareable link will be generated below.\n")

# Create and launch the app
multiagent_app = create_multiagent_gradio_app()
multiagent_app.launch(
    share=True,           # Create public link
    show_error=True,      # Show errors in UI
    quiet=False           # Show startup messages
)

## 🚀 Deployment

### Hugging Face Spaces
1. Create Space at [huggingface.co/spaces](https://huggingface.co/spaces) with **Gradio** SDK
2. Add `GEMINI_API_KEY` as secret
3. Upload notebook or create `app.py`

### Local Setup
```bash
pip install -r requirements.txt
echo "GEMINI_API_KEY=your-key" > .env
jupyter notebook gemini_polyglot_guardian.ipynb
```

---

**Built with ❤️ for the Hackathon** | MIT License